# Script 2: Filter Articles Using `semantic.predicate()`

Inference is expensive, so before running feature extraction or annotation, we filter out articles that are unlikely to belong to the `"artificial-intelligence"` slug.

We define a natural language description of what qualifies as an AI-related article. Then, using `semantic.predicate()`, we evaluate each article against this description. Downstream, only articles that return `True` are kept; the rest are excluded from the dataset. This helps focus downstream work on relevant content.

In [ ]:
import fenic as fc
from dotenv import load_dotenv

load_dotenv()

fc.configure_logging()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)

In [2]:
cleaned = session.table("cleaned")

## Step 1: Define prompt to identify articles that are correctly tagged as `artificial-intelligence`

In [3]:
is_on_topic_instruction = """
This article is about artificial intelligence, machine learning, or AI-related tools and applications. It fits this category based on its title: {title} and body: {text}.

This includes content such as:
- AI/ML tutorials, guides, or technical explanations
- AI tools and platforms (e.g., ChatGPT, Claude, Copilot)
- AI coding assistants and development workflows
- Applications of AI in business, creativity, or other domains
- AI ethics, safety, and societal impact
- Machine learning concepts, techniques, or research
- Industry news, trends, or analysis related to AI
- Personal experiences using AI tools productively

It is **not** about AI if it instead focuses on:
- General tech topics without an AI focus
- Personal stories, relationships, or politics unrelated to AI
- General software development or programming topics
- Business or entrepreneurship content lacking an AI angle
- Fiction, poetry, or creative writing without AI themes
- Lifestyle, travel, health, or historical topics
- Promotional clickbait with no real AI content
- Superficial “get rich with AI” schemes

Edge cases should **still be considered AI-related** if the article:
- Discusses automation, intelligent systems, or algorithmic decision-making
- Covers AI’s impact on specific industries or professions
- Explores philosophical or ethical questions about intelligence or human-AI interaction
"""

## Step 2: Run `semantic.predicate()` to label whether an article is on topic or not

In [ ]:
with_on_topic_annotation = (
    cleaned
    .with_column("is_on_topic", fc.semantic.predicate(is_on_topic_instruction))
    .cache()
)

## Step 3: Save Results

In [ ]:
with_on_topic_annotation.write.save_as_table("with_on_topic_label", mode="error")

In [ ]:
with_on_topic_annotation.filter(~fc.col("is_on_topic")).select("title", "text").show(100)

In [8]:
session.stop()